# Rotating Convection Shell
#### Attempting to code a rotationg shell with convection, so that i may be able to modify it to be similar to the suns.
Eiganvalue problem
using equations of https://www.astro.physik.uni-potsdam.de/~seehafer/TUTORIALS.DIR/Boussinesq_spherical_shell.pdf

Non dimesional equations
#### $ { E \biggl( {\frac{ \partial v }{ \partial t } +( v \cdot \nabla)v - \nabla^2 v} \biggr) }  = { -2e_z  \times v - \nabla p + RaT \frac{ r }{ R_o } } $
#### $ \frac{\partial T}{\partial t} +v \cdot \nabla T = \frac{1}{Pr} \nabla^2 T $
#### $ \nabla \cdot v = 0 $

##### $ E = \frac{v}{D^2 \Omega} $
##### $ Ra = \frac{\alpha \delta T g_0 D}{ \Omega v} $
##### $ Pr = \frac{\nu}{\kappa} $
##### $ D = R_o - R_i $

E : Ekman number  
v : velocity of a fluid parcel  
t : time  
e_z : unit vector in z direction  
p : pressure  
Ra = Modified version of the Rayleigh number  
T : Temperature  
r: position vector  
R_o : outer radius  
R_i : inner raduis  
$ \nu $ : kinematic viscosity  
$ \kappa $ : Thermal Diffusivity  
Pr : Prandtl number  

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import dedalus.public as d3
import scipy.constants as sc
import logging
logger = logging.getLogger(__name__)

coordinate system and fields.

In [2]:
stop_sim_time = 30
timestepper = d3.RK111
timestep = 0.5

In [3]:
#Working on assumption angular mimentum is constant
Omega = 2
R_i = 1
R_o = 10

In [4]:
coord = d3.SphericalCoordinates("r", "theta", "phi")
dist = d3.Distributor(coord, dtype = np.complex128)
shell = d3.ShellBasis(coord, shape = ( 128, 128, 128 ),  radii = ( R_i, R_o ), dealias = 3/2, dtype = np.complex128)
surface = shell.outer_surface
r, theta, phi = dist.local_grids(shell)


eig = dist.Field( name = "eig" )
v = dist.VectorField( coord, name = "v", bases = shell )
p = dist.Field( name = "p", bases = shell )
r = dist.Field( name = "r", bases = shell )
e_z = dist.Field( name = "e_z" ,bases = shell )
#E = dist.Field( name = "E", bases = shell )

tau = dist.Field( name = "tau" )
tau1 = dist.Field( name = "tau1")

In [5]:
#dt = lambda A: d3.Differentiate(A, coord["t"],1)
#lift_basis = shell.derivative_basis(1)
#lift = lambda A: d3.lift(A, lift_basis, -1)

grad = lambda A: d3.Gradient(A).evaluate()
div = lambda A:  d3.Divergence(A).evaluate()
lap = lambda A: d3.Laplacian(A).evaluate()


Creating problem and adding equations.

In [6]:


#alpha: thermal expansion coeffiecent
alpha = 2
#nu: kinematic viscosity
nu = 3
#kappa : thermal diffusivity
kappa = 4

In [7]:
#problem = d3.EVP([v, p, r, tau, tau1], eigenvalue = eig, namespace = locals())
problem = d3.IVP([v, p, r, tau, tau1], namespace = locals())

#problem.add_equation("E  =  v / (D**2 * Omega)")
D = R_o - R_i
E = v/(D**2 *Omega)

P_r = nu/kappa
#problem.add_equation("
problem.add_equation("E * (dt(v) + v**2 + grad(v) - lap(v)) + (2 * e_z * Cross(v)) + grad(p) - ((R_a * T * r) / R_o) = 0")
problem.add_equation("dt(T) + v *dot(grad(T) = (1/P_r) * lap(T)")
problem.add_equation("div(v) = 0 ")

/home/ben2121/miniforge3/envs/dedalus3/lib/python3.13/site-packages/dedalus/core/problems.py:118: SyntaxWarning: invalid escape sequence '\ '
  """
/home/ben2121/miniforge3/envs/dedalus3/lib/python3.13/site-packages/dedalus/core/problems.py:191: SyntaxWarning: invalid escape sequence '\ '
  """


ValueError: Objects are not all equal.